In [ ]:
#===================================================================================\
# Count number of distinct txs in logs
#===================================================================================\\


import os
import re

# ---------------------------------------------------------
# STEP 1: CONFIGURATION
# ---------------------------------------------------------
# Path to the directory containing the log files
# Use the variable appropriate for your test case
empty_blocks = '/home/christos/IOHK/repos/partner-chains/e2e-tests/utils/logs/from_2026-01-15_17-35-00_to_2026-01-15_17-50-00/'
test_490_txs = '/home/christos/shielded/jupyter/2026-01-14/all_logs/'
new_100_txs = '/home/christos/IOHK/repos/partner-chains/e2e-tests/utils/logs/from_2026-01-15_21-30-50_to_2026-01-15_21-34-35/'
new_200_txs = '/home/christos/IOHK/repos/partner-chains/e2e-tests/utils/logs/from_2026-01-15_21-38-08_to_2026-01-15_21-44-01/'
txs_91_investigating_io = '/home/christos/IOHK/repos/partner-chains/e2e-tests/utils/benchmarks/logs/from_2026-01-20_17-17-16_to_2026-01-20_17-21-40/'

# Set active log directory here
log_dir = test_490_txs

RELAYS = ["ferdie", "george", "henry", "iris", "jack", "paul", "quinn", "rita", "sam", "tom"]
BLOCK_PRODUCERS = ["alice", "bob", "charlie", "dave", "eve", "kate", "leo", "mike", "nina", "oliver"]
NODES = RELAYS + BLOCK_PRODUCERS

# ---------------------------------------------------------
# STEP 2: DEFINE PARSING LOGIC
# ---------------------------------------------------------

def count_validated_transactions(log_directory, nodes):
    """
    Reads log files and counts unique 'Validated Midnight transaction' entries
    using regex. Returns a set of all unique TXs and a dict of counts per node.
    """
    total_unique_txs = set()
    node_stats = {}
    files_processed = 0

    # Regex matches: Validated Midnight transaction "618804..."
    tx_pattern = re.compile(r'Validated Midnight transaction "([a-fA-F0-9]+)"')

    print(f"Scanning logs in: {log_directory}\n")

    for node in nodes:
        file_path = os.path.join(log_directory, f"{node}.txt")
        node_unique_txs = set()
        
        if not os.path.exists(file_path):
            # If file doesn't exist, we just record 0 for this node
            node_stats[node] = 0
            # print(f"⚠️ Log file not found for: {node}") # Optional warning
            continue

        try:
            with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                for line in f:
                    match = tx_pattern.search(line)
                    if match:
                        tx_hash = match.group(1)
                        node_unique_txs.add(tx_hash)
            
            # Update stats
            node_stats[node] = len(node_unique_txs)
            total_unique_txs.update(node_unique_txs)
            files_processed += 1
            
        except Exception as e:
            print(f"Error reading file for {node}: {e}")
            node_stats[node] = 0

    return total_unique_txs, node_stats, files_processed

# ---------------------------------------------------------
# STEP 3: RUN ANALYSIS
# ---------------------------------------------------------

unique_tx_set, node_counts, files_count = count_validated_transactions(log_dir, NODES)

# Print per-node statistics
for node in NODES:
    count = node_counts.get(node, 0)
    print(f"{node}: {count} distinct Validated transactions")

print("\n" + "=" * 50)
print(f"TOTAL DISTINCT TRANSACTIONS ACROSS ALL NODES: {len(unique_tx_set)}")
print("=" * 50)

In [ ]:
#===================================================================================\
# Plot block production timeline
#===================================================================================\\


import matplotlib.dates as mdates
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import re
import os
from datetime import datetime

# ---------------------------------------------------------
# STEP 4: PARSE BLOCK CREATION TIMES (Re-parsing for Timeline)
# ---------------------------------------------------------
# We need to scan the logs again specifically for block creation events 
# to build the creation_registry required for the plot.

creation_registry = {} 

# Regex to capture timestamp and creation event
timestamp_pattern = re.compile(r"^(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}\.\d+)")
creation_pattern = re.compile(r"Pre-sealed block for proposal at (\d+)")

print(f"Scanning for block creation events in: {log_dir}")

for node in NODES:
    filepath = os.path.join(log_dir, f"{node}.txt")
    
    if not os.path.exists(filepath):
        continue
        
    with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            line = line.strip()
            
            # Check for creation line first (optimization)
            create_match = creation_pattern.search(line)
            if create_match:
                # Extract Timestamp
                ts_match = timestamp_pattern.search(line)
                if not ts_match: continue
                
                try:
                    ts_str = ts_match.group(1)
                    if len(ts_str.split('.')[-1]) > 6: ts_str = ts_str[:26]
                    current_time = datetime.strptime(ts_str, "%Y-%m-%d %H:%M:%S.%f")
                    
                    block_num = int(create_match.group(1))
                    
                    # Store data
                    creation_registry[block_num] = {
                        'time': current_time,
                        'creator': node
                    }
                except ValueError:
                    continue

# ---------------------------------------------------------
# STEP 5: PLOT BLOCK CREATION TIMELINE
# ---------------------------------------------------------

if not creation_registry:
    print("No block creation events found. Check log path.")
else:
    # Convert registry to DataFrame
    creation_data = []
    for block_num, info in creation_registry.items():
        creation_data.append({
            "Block": block_num,
            "Timestamp": info['time'],
            "Creator": info['creator']
        })

    df_creation = pd.DataFrame(creation_data).sort_values("Block")

    print(f"Found {len(df_creation)} blocks. Plotting...")

    # Setup the Plot
    plt.figure(figsize=(14, 7))
    sns.set_theme(style="whitegrid")

    # 1. Draw a line to show the progression
    sns.lineplot(
        data=df_creation,
        x="Block",
        y="Timestamp",
        color="gray",
        alpha=0.3,
        zorder=1
    )

    # 2. Draw points colored by the Creator Node
    plot = sns.scatterplot(
        data=df_creation,
        x="Block",
        y="Timestamp",
        hue="Creator",
        palette="tab20",  # Colorful palette for many nodes
        s=100,            # Dot size
        edgecolor="black",
        zorder=2
    )

    # Format Y-Axis to show HH:MM:SS
    plot.yaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))

    plt.title("Block Creation Timeline", fontsize=16)
    plt.xlabel("Block Number", fontsize=12)
    plt.ylabel("Creation Time (UTC)", fontsize=12)

    # Ensure integer ticks on X-axis if valid range
    if df_creation["Block"].nunique() < 20:
        plt.xticks(df_creation["Block"].unique())

    # Move legend outside
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0., title="Producer")
    
    plt.tight_layout()
    plt.savefig("block_creation_timeline.png", dpi=300)
    print("Graph saved as 'block_creation_timeline.png'")
    
    plt.show()

In [ ]:
#===================================================================================
# Plot all Block Producer's block production time and the start delay
#===================================================================================

files_to_process = []
for node in BLOCK_PRODUCERS:
    # os.path.join automatically adds the '/' if log_dir doesn't have it
    files_to_process.append(os.path.join(log_dir, f"{node}.txt"))

# # ---------------------------------------------------------
# # STEP 6: PARSE THE LOGS
# # ---------------------------------------------------------

def parse_substrate_logs_robust(file_list):
    data = []
    
    # 1. Regex to capture Timestamp and the rest of the message
    # Matches: 2026-01-15 17:39:36.048  INFO ...
    base_pattern = re.compile(r"^(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}\.\d{3})\s+(.*)")
    
    # 2. Regex for START event (captures Parent Block #)
    # Matches: ... Starting consensus session on top of parent ... (#14406)
    start_pattern = re.compile(r"Starting consensus session on top of parent .* \(#(\d+)\)")
    
    # 3. Regex for END event (captures Current Block #)
    # Matches: ... Prepared block for proposing at 14407 ...
    end_pattern = re.compile(r"Prepared block for proposing at (\d+)")

    for filename in file_list:
        node_name = os.path.splitext(os.path.basename(filename))[0].capitalize()
        
        # Dictionary to store timestamps for each block
        # Structure: { 14407: {'start': datetime, 'end': datetime}, ... }
        block_events = {}
        
        with open(filename, 'r') as f:
            for line in f:
                line = line.strip()
                if not line: continue
                
                # Extract Timestamp
                base_match = base_pattern.search(line)
                if not base_match: continue
                
                timestamp_str = base_match.group(1)
                message = base_match.group(2)
                
                try:
                    current_time = datetime.strptime(timestamp_str, "%Y-%m-%d %H:%M:%S.%f")
                except ValueError:
                    continue

                # --- CHECK FOR START (Parent Block) ---
                start_match = start_pattern.search(message)
                if start_match:
                    parent_block = int(start_match.group(1))
                    # If we are working on top of 14406, we are building 14407
                    target_block = parent_block + 1
                    
                    if target_block not in block_events: block_events[target_block] = {}
                    block_events[target_block]['start'] = current_time
                    continue

                # --- CHECK FOR END (Current Block) ---
                end_match = end_pattern.search(message)
                if end_match:
                    target_block = int(end_match.group(1))
                    
                    if target_block not in block_events: block_events[target_block] = {}
                    block_events[target_block]['end'] = current_time
                    continue

        # --- CALCULATE DURATIONS ---
        # Now we iterate through the dictionary and calculate stats for blocks that have both times
        for block_num, times in block_events.items():
            if 'start' in times and 'end' in times:
                start_time = times['start']
                end_time = times['end']
                
                # A. Production Time
                # Calculate diff in milliseconds
                duration_ms = (end_time - start_time).total_seconds() * 1000
                
                # Sanity check: If duration is negative (rare clock skew?), skip
                if duration_ms < 0: continue

                # B. Start Delay (Ideal Target vs Actual Start)
                # Calculate seconds past the minute
                total_seconds = start_time.second + (start_time.microsecond / 1e6)
                remainder = total_seconds % 6 
                start_delay_ms = remainder * 1000
                
                data.append({
                    "Node": node_name,
                    "Block": block_num,
                    "ProductionTime_ms": duration_ms,
                    "StartDelay_ms": start_delay_ms
                })

    return pd.DataFrame(data)

# Run the parser
df = parse_substrate_logs_robust(files_to_process)

print(f"Parsed {len(df)} blocks successfully.")
if not df.empty:
    print(df.head())
else:
    print("No matching blocks found. Check log paths or regex.")

# ---------------------------------------------------------
# STEP 3: PLOT THE GRAPHS
# ---------------------------------------------------------

if not df.empty:
    fig, axes = plt.subplots(2, 1, figsize=(12, 10), sharex=True)
    sns.set_theme(style="whitegrid")

    # GRAPH 1: Production Time (Duration)
    # How long did the CPU actually churn to build the block?
    sns.lineplot(
        ax=axes[0],
        data=df, 
        x="Block", 
        y="ProductionTime_ms", 
        hue="Node", 
        marker="o",
        palette="tab10"
    )
    axes[0].set_title("1. Actual Block Production Duration (End - Start)", fontsize=14, fontweight='bold')
    axes[0].set_ylabel("Duration (ms)")
    axes[0].legend(loc='upper right')

    # GRAPH 2: Start Delay
    # How long after the ideal 6s slot did the node start working?
    sns.lineplot(
        ax=axes[1],
        data=df, 
        x="Block", 
        y="StartDelay_ms", 
        hue="Node", 
        marker="s", # Square marker to distinguish
        palette="tab10"
    )
    axes[1].set_title("2. Start Delay (Actual Start - Target Slot Time)", fontsize=14, fontweight='bold')
    axes[1].set_ylabel("Delay (ms)")
    axes[1].set_xlabel("Block Number")
    
    # Force Integer X-Axis
    if len(df) < 20:
        axes[1].set_xticks(sorted(df['Block'].unique()))

    plt.tight_layout()
    plt.savefig("block_timing_analysis.png")
    plt.show()
else:
    print("No matching log lines found. Check regex patterns.")

In [ ]:
#===================================================================================
# Plot each Block Producer's block production time
#===================================================================================

# 1. Get the list of nodes
unique_nodes = sorted(df['Node'].unique())

# 2. Determine global X-axis limits
min_block = df['Block'].min()
max_block = df['Block'].max()

# 3. Determine global Y-axis limit (New Step)
# Find the max time across ALL nodes and add a 10% buffer for visual spacing
max_y_val = df['ProductionTime_ms'].max()
global_y_limit = max_y_val * 1.1 

# 4. Iterate and plot
for node in unique_nodes:
    # Filter data for this specific node
    node_data = df[df['Node'] == node]
    
    # Skip if no data for this node
    if node_data.empty:
        continue

    # Create a new figure for each node
    plt.figure(figsize=(10, 3))
    
    # Plot Production Time
    sns.lineplot(
        data=node_data, 
        x="Block", 
        y="ProductionTime_ms", 
        marker="o", 
        color="#2ca02c"
    )
    
    # Formatting
    plt.title(f"Production Time (End - Start) for: {node}", fontsize=14, fontweight='bold')
    plt.ylabel("Time (ms)")
    plt.xlabel("Block Number")
    
    # --- GLOBAL AXIS LIMITS ---
    # 1. Apply global X limits
    plt.xlim(min_block, max_block)
    
    # 2. Apply global Y limits (0 to Global Max)
    plt.ylim(0, global_y_limit)
    # --------------------------
    
    # Handle ticks: If the total range is small, show every integer. 
    if (max_block - min_block) < 50:
        plt.xticks(range(int(min_block), int(max_block) + 1))
        
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.show()

In [ ]:
# ---------------------------------------------------------
# Plot all block import times
# ---------------------------------------------------------




# ---------------------------------------------------------
# STEP 2: HELPER FUNCTIONS
# ---------------------------------------------------------

def to_short_hash(full_hash):
    """
    Converts a full hash (e.g. 0xf2d38b...ed171) to the log format (0xf2d3…d171).
    Standard Substrate log format is usually: 0x + first 4 + … + last 4
    """
    if not full_hash.startswith("0x"):
        return full_hash
    
    # Remove 0x prefix for slicing
    clean = full_hash[2:]
    if len(clean) < 8: return full_hash
    
    return f"0x{clean[:4]}…{clean[-4:]}"

# ---------------------------------------------------------
# STEP 3: PARSE THE FILES
# ---------------------------------------------------------

# Dictionary to store creation times keyed by the BLOCK HASH (Short format)
# Key: ShortHash (str), Value: { 'time': datetime, 'block_num': int, 'node': str }
creation_registry = {}

# List to store raw import events to be processed later
# Item: { 'time': datetime, 'block_num': int, 'short_hash': str, 'node': str }
import_events = []

# Regex patterns
timestamp_pattern = re.compile(r"^(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}\.\d+)")

# Capture Creation: "Pre-sealed block for proposal at 3040. Hash now 0xf2d3..."
# We use the "Pre-sealed" line because it contains the final finalized hash.
creation_pattern = re.compile(r"Pre-sealed block for proposal at (\d+)\. Hash now (0x[a-fA-F0-9]+)")

# Capture Import: "Imported #3040 (0x4e7c…cc64 → 0xf2d3…d171)"
# We capture Group 1 (Block Num) and Group 2 (The second hash, which is the new block hash)
import_pattern = re.compile(r"Imported #(\d+) \(.*? → (0x[a-fA-F0-9…\.]+)\)")

print(f"Reading logs from: {log_dir}")

for node in NODES:
    filepath = os.path.join(log_dir, f"{node}.txt")
    
    if not os.path.exists(filepath):
        print(f"  - Warning: Log file not found for {node}")
        continue
        
    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if not line: continue
            
            # 1. Extract Timestamp
            ts_match = timestamp_pattern.search(line)
            if not ts_match: continue
            
            try:
                ts_str = ts_match.group(1)
                # Truncate microseconds if too long (Python limit)
                if len(ts_str.split('.')[-1]) > 6: ts_str = ts_str[:26] 
                current_time = datetime.strptime(ts_str, "%Y-%m-%d %H:%M:%S.%f")
            except ValueError:
                continue 

            # 2. Check for BLOCK CREATION (The Source of Truth)
            create_match = creation_pattern.search(line)
            if create_match:
                block_num = int(create_match.group(1))
                full_hash = create_match.group(2)
                short_hash = to_short_hash(full_hash)
                
                # We store this hash as the canonical creation event
                creation_registry[short_hash] = {
                    'time': current_time,
                    'block_num': block_num,
                    'creator': node
                }
                continue

            # 3. Check for BLOCK IMPORT
            import_match = import_pattern.search(line)
            if import_match:
                block_num = int(import_match.group(1))
                imported_hash = import_match.group(2)
                
                import_events.append({
                    'time': current_time,
                    'block_num': block_num,
                    'short_hash': imported_hash,
                    'node': node
                })

print(f"Processing complete.")
print(f"  - Unique Blocks Created: {len(creation_registry)}")
print(f"  - Total Import Events: {len(import_events)}")

# ---------------------------------------------------------
# STEP 4: CALCULATE LATENCY (MATCHING BY HASH)
# ---------------------------------------------------------

data = []

for event in import_events:
    target_hash = event['short_hash']
    importer = event['node']
    import_time = event['time']
    
    # Only calculate if we have a creation record for THIS SPECIFIC HASH
    if target_hash in creation_registry:
        genesis_info = creation_registry[target_hash]
        creation_time = genesis_info['time']
        creator = genesis_info['creator']
        block_num = genesis_info['block_num']
        
        # If this node is the creator, latency is 0
        if importer == creator:
            delay_ms = 0.0
        else:
            delta = import_time - creation_time
            delay_ms = delta.total_seconds() * 1000.0
        
        data.append({
            "Block": block_num,
            "Node": importer,
            "ImportTime_ms": delay_ms,
            "Hash": target_hash
        })

# ---------------------------------------------------------
# STEP 5: PLOT
# ---------------------------------------------------------

if not data:
    print("No matching hash data found! Check if the log format matches the patterns.")
else:
    df = pd.DataFrame(data)
    df = df.sort_values(by="Block")
    
    # Display sample to verify
    print("\nSample Data (First 5 rows):")
    print(df.head())

    plt.figure(figsize=(14, 7))
    sns.set_theme(style="whitegrid")

    sns.lineplot(
        data=df, 
        x="Block", 
        y="ImportTime_ms", 
        hue="Node", 
        marker="o",
        palette="tab20", 
        linewidth=1.5,
        errorbar=None 
    )

    plt.title("Block Propagation Delay", fontsize=16)
    plt.xlabel("Block Number", fontsize=12)
    plt.ylabel("Propagation Delay (ms)", fontsize=12)
    
    # Formatting X-axis
    if df["Block"].nunique() < 20:
        plt.xticks(df["Block"].unique())

    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0., title="Node")
    plt.tight_layout()
    
    plt.savefig("import_times_hash_matched.png", dpi=300)
    print("\nGraph saved as 'import_times_hash_matched.png'")
    
    plt.show()

In [ ]:
# ---------------------------------------------------------
# Plot block import times per node
# ---------------------------------------------------------


# 1. PRE-CALCULATE THE Y-AXIS LIMIT
# Find the max value in the whole dataset and add a 10% buffer for visual spacing
max_val = df['ImportTime_ms'].max()
global_y_limit = max_val * 1.1 

# Get a list of all unique nodes and sort them alphabetically
nodes = sorted(df['Node'].unique())

# Loop through each node and create a separate graph
for node in nodes:
    # Filter the data for just this node
    node_data = df[df['Node'] == node]
    
    # Create a new figure for this node
    plt.figure(figsize=(12, 3)) 
    
    # Plot the line
    sns.lineplot(
        data=node_data, 
        x="Block", 
        y="ImportTime_ms", 
        marker="o",
        color='tab:blue',
        errorbar=None
    )
    
    # Formatting
    plt.title(f"Import Times for: {node.upper()}", fontsize=14, fontweight='bold')
    plt.ylabel("Time (ms)")
    plt.xlabel("Block Number")
    
    # --- CRITICAL CHANGE HERE ---
    # Force every plot to use the same 0 to Global Max scale
    plt.ylim(0, global_y_limit)
    # ----------------------------
    
    # Ensure X-axis shows every block integer (if list isn't too long)
    if len(df["Block"].unique()) < 50:
        plt.xticks(df["Block"].unique())
    
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    
    # Display the graph
    plt.show()